In [1]:
import os, gc, importlib.util
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
VOL = {"v1": "/tmp/vol_test", "v2": "/tmp/vol_test_v2"}

FILES = {}
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    for f in files:
        if not f.endswith(".npz"):
            FILES.setdefault(f, os.path.join(root, f))

def find_file(name):
    if name not in FILES:
        raise FileNotFoundError(name)
    return FILES[name]

def load_module(name, fname):
    path = find_file(fname)
    print(f"{fname:28s} -> {path}")
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

# nama: (backbone, file bobot, versi data)
DINO = "vit_small_patch14_dinov2.lvd142m"
MODELS = {
    "dinov2_student_v3": (DINO, ["dinov2s_student_v3_best.pt"], "v2"),
    "dinov2_student_v2": (DINO, ["dinov2s_student_v2_best.pt"], "v2"),
    "student_v3":        ("efficientnet_b0", ["effb0_student_v3_best.pt"], "v2"),
    "student_v3_s7":     ("efficientnet_b0", ["effb0_student_v3_s7_best.pt"], "v2"),
}
USE = ["dinov2_student_v3", "dinov2_student_v2", "student_v3", "student_v3_s7"]   # val 0.8777 | gold 0.8632

need = sorted({MODELS[n][2] for n in USE})
PP = {}
if "v1" in need:
    PP["v1"] = load_module("pp1", "03_preprocess_mri.py")      # preprocessing v1 (3 view, 224)
if "v2" in need:
    PP["v2"] = load_module("pp2", "03b_preprocess_mri_v2.py")  # preprocessing v2 (5 view, 256)
tb = load_module("tb", "09_train_student.py")                  # KneeNet: ViT otomatis resize 256 -> 224
LABELS = tb.LABELS

for name in USE:
    for w in MODELS[name][1]:
        print(f"{name:18s} {w:32s} -> {find_file(w)}")
print("Data yang perlu diproses:", need, "| GPU:", torch.cuda.device_count())

03b_preprocess_mri_v2.py     -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-02b-preprocess-mri-v2/03b_preprocess_mri_v2.py
09_train_student.py          -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-10-student-v3/09_train_student.py
dinov2_student_v3  dinov2s_student_v3_best.pt       -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-10-student-v3/dinov2s_student_v3_best.pt
dinov2_student_v2  dinov2s_student_v2_best.pt       -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-08-student-dinov2-v2data/dinov2s_student_v2_best.pt
student_v3         effb0_student_v3_best.pt         -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-10-student-v3/effb0_student_v3_best.pt
student_v3_s7      effb0_student_v3_s7_best.pt      -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-10-student-v3/effb0_student_v3_s7_best.pt
Data yang perlu diproses: ['v2'] | GPU: 2


In [2]:
series = pd.read_csv(f"{COMP}/test_series.csv")
test = pd.read_csv(f"{COMP}/test.csv")
uids = test["StudyInstanceUID"].tolist()

avail = {}
for ver in need:
    pp = PP[ver]
    pp.SPLIT, pp.OUT_DIR = "test", VOL[ver]
    os.makedirs(VOL[ver], exist_ok=True)
    failed = 0
    for uid, g in tqdm(series.groupby("StudyInstanceUID"), desc=f"Preprocess {ver}"):
        try:
            pp.process_study(uid, g)
        except Exception as e:
            failed += 1
            print("Gagal:", ver, uid, type(e).__name__, e)
    avail[ver] = [u for u in uids if os.path.exists(f"{VOL[ver]}/{u}.npz")]
    print(f"{ver}: volume tersedia {len(avail[ver])}/{len(uids)} | gagal {failed}")
    assert len(avail[ver]) >= 0.95 * len(uids), f"Terlalu banyak studi gagal ({ver}), jangan disubmit"

Preprocess v2:   0%|          | 0/3 [00:00<?, ?it/s]

v2: volume tersedia 3/3 | gagal 0


In [3]:
member_preds = {}
for name in USE:
    backbone, wfiles, ver = MODELS[name]
    ok = avail[ver]
    n_views = int(np.load(f"{VOL[ver]}/{ok[0]}.npz")["mask"].shape[0])
    ds = tb.KneeDS(ok, np.full((len(ok), len(LABELS)), np.nan, np.float32), VOL[ver], train=False)
    dl = torch.utils.data.DataLoader(ds, batch_size=2, shuffle=False, num_workers=0)

    ps = []
    for w in wfiles:
        net = tb.KneeNet(backbone, n_views=n_views, pretrained=False)   # struktur saja, tanpa download
        net.load_state_dict(torch.load(find_file(w), map_location="cpu"))
        net = net.cuda().eval()
        if "vit" not in backbone:
            net = net.to(memory_format=torch.channels_last)
        ps.append(tb.predict(net, dl))
        del net; gc.collect(); torch.cuda.empty_cache()

    p = pd.DataFrame(np.mean(ps, axis=0), columns=LABELS)
    p.insert(0, "StudyInstanceUID", ok)
    member_preds[name] = test[["StudyInstanceUID"]].merge(p, on="StudyInstanceUID", how="left")[LABELS].values
    print(f"{name:18s} (data {ver}, {n_views} view) selesai | "
          f"rata-rata prediksi {np.nanmean(member_preds[name]):.3f}")

dinov2_student_v3  (data v2, 5 view) selesai | rata-rata prediksi 0.231
dinov2_student_v2  (data v2, 5 view) selesai | rata-rata prediksi 0.189
student_v3         (data v2, 5 view) selesai | rata-rata prediksi 0.265
student_v3_s7      (data v2, 5 view) selesai | rata-rata prediksi 0.247


In [4]:
p_ens = np.nanmean(np.stack([member_preds[n] for n in USE]), axis=0)
for n in USE:
    pd.DataFrame(member_preds[n], columns=LABELS).assign(StudyInstanceUID=uids) \
      .to_csv(f"/kaggle/working/test_preds_{n}.csv", index=False)
print("Anggota ensemble:", USE, "| bentuk:", p_ens.shape)

sub = pd.DataFrame(p_ens, columns=LABELS)
sub.insert(0, "StudyInstanceUID", uids)
n_missing = sub[LABELS].isna().any(axis=1).sum()
sub[LABELS] = sub[LABELS].fillna(0.5)

sample = pd.read_csv(f"{COMP}/sample_submission.csv")
sub = sub[sample.columns]
sub.to_csv("/kaggle/working/submission.csv", index=False)

print("Studi tanpa prediksi (diisi 0.5):", n_missing)
print("Shape submission:", sub.shape, "| shape sample:", sample.shape)
print(sub.head())

Anggota ensemble: ['dinov2_student_v3', 'dinov2_student_v2', 'student_v3', 'student_v3_s7'] | bentuk: (3, 12)
Studi tanpa prediksi (diisi 0.5): 0
Shape submission: (3, 13) | shape sample: (3, 13)
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.012510  0.056286   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.368870  0.111129   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.003115  0.041142   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.354902          0.058961   0.216102    0.112442  0.240260   
1         0.485267          0.287255   0.312842    0.231231  0.287986   
2         0.446622          0.084538   0.467829    0.190639  0.751862   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.525031   0.063409  0.683521   0.122910  0.005031  
1  0.322654   0.544818  0.144040   0.406225  0.096141  
2  0.106918   0.042019  0.165018   0.0